# YOLOv8m + MLCA (Colony-YOLO) sobre el subdataset v5

Prueba: el modelo de Belén (**YOLOv8m pre-entrenado en COCO**, su **dataset v5** y sus **hiperparámetros**) agregando el mecanismo de atención del paper Colony-YOLO: **C2f-MLCA en el neck** (MLCA dentro de cada Bottleneck, Figura 6).

Se entrenan dos variantes con la misma configuración, para que la única diferencia sea el MLCA:

| variante | qué es |
|---|---|
| `base` | YOLOv8m tal cual (control) |
| `mlca` | YOLOv8m con los 4 C2f del neck → C2f-MLCA (8 MLCA). El backbone no cambia |

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno → GPU T4`. Después ejecutá las celdas en orden.

Tiempo: ~13 min por entrenamiento en una T4. Con 3 seeds × 2 variantes ≈ 1 h 20 min, más la evaluación. Todo se guarda en Drive: si Colab se desconecta, volvé a ejecutar las celdas en orden y el entrenamiento **saltea las corridas ya terminadas**.

## 1 · GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2 · Librerías
Misma versión de ultralytics que usó Belén (8.4.163).

In [ ]:
!pip install -q ultralytics==8.4.163 roboflow
import ultralytics
print('ultralytics', ultralytics.__version__)

## 3 · Configuración

La API key de Roboflow **no** se escribe en el notebook. La celda la busca en los *Secrets* de Colab (ícono de llave 🔑 a la izquierda, nombre `ROBOFLOW_API_KEY`) y, si no está, la pide por teclado.

In [ ]:
SEEDS     = [0, 1, 2]          # una corrida por seed y por variante
VARIANTES = ['base', 'mlca']
BATCH     = 32                 # el de Belén (entra en los 15 GB de la T4)
EPOCHS    = 100
DRIVE_DIR = '/content/drive/MyDrive/yolov8m_mlca'

# Hiperparámetros de su best_v8.pt (train_args)
HER_ARGS = dict(patience=0, imgsz=640, optimizer='Adam', lr0=0.0001, lrf=0.01, momentum=0.937,
                weight_decay=0.0001, warmup_epochs=3.0, warmup_momentum=0.8, warmup_bias_lr=0.1,
                box=7.5, cls=0.5, dfl=1.5, nbs=64, hsv_h=0.0, hsv_s=0.0, hsv_v=0.15, degrees=12.0,
                translate=0.1, scale=0.1, shear=0.0, perspective=0.0, flipud=0.0, fliplr=0.5,
                mosaic=1.0, mixup=0.0, copy_paste=0.0, close_mosaic=10, erasing=0.4, cos_lr=False,
                deterministic=True, amp=True, iou=0.7, max_det=300, pretrained=True, workers=8)

## 4 · Google Drive

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs(DRIVE_DIR, exist_ok=True)
print('resultados en', DRIVE_DIR)

## 5 · Dataset (subdataset v5, el que usó Belén)

In [ ]:
import os, getpass
from roboflow import Roboflow

try:
    from google.colab import userdata
    API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    API_KEY = None
if not API_KEY:
    API_KEY = getpass.getpass('API key de Roboflow: ')

rf = Roboflow(api_key=API_KEY.strip())
ds = (rf.workspace('ucuestudiantesproyecto-gmail-com')
        .project('subdataset_v2-ucu-udelar-2026-a').version(5).download('yolov8', location='/content/dataset_v5'))
del API_KEY

DATA_YAML = '/content/data_v5.yaml'
with open(DATA_YAML, 'w') as f:
    f.write('path: /content/dataset_v5\ntrain: train/images\nval: valid/images\ntest: test/images\n'
            "nc: 3\nnames: ['col', 'osa', 'sau']\n")
for s in ('train', 'valid', 'test'):
    print(s, len(os.listdir(f'/content/dataset_v5/{s}/images')), 'fotos')   # esperado: 102 / 24 / 31

## 6 · C2f-MLCA (atención del paper)

* **MLCA** (Mixed Local Channel Attention): atención de canales local (pooling 5×5) + global, cada una con un Conv1d de kernel *k* según la Ec. (3) del paper (g = b = 2).
* **Bottleneck_MLCA**: `x → Conv → Conv → MLCA (→ + x si hay shortcut)`. La atención modula la rama residual (Figura 6).
* **C2f_MLCA**: un C2f cuyos Bottleneck son Bottleneck_MLCA.

ultralytics solo calcula canales y repeticiones para los módulos de una lista interna. Durante la construcción del modelo, el nombre `C2fPSA` (que está en esa lista y YOLOv8m no usa) apunta a `C2f_MLCA`. Así solo llevan atención las capas que el YAML declara como `C2f_MLCA`, y el backbone queda intacto.

**Si se reinicia el entorno, ejecutá esta celda antes de cargar un modelo con MLCA.**

In [ ]:
import math, inspect
import torch
import torch.nn as nn
import torch.nn.functional as F
import ultralytics.nn.tasks as tasks
import ultralytics.nn.modules.block as block
from ultralytics.nn.modules.block import C2f, Bottleneck


class MLCA(nn.Module):
    def __init__(self, channel, local_size=5, gamma=2, b=2, local_weight=0.5):
        super().__init__()
        self.local_size = int(local_size)
        self.local_weight = float(local_weight)
        c = int(channel)
        if c % 2 == 0:                               # Ec. (3): si C es par, C+1
            c += 1
        t = int(abs(math.log2(c) / gamma + b / gamma))
        k = max(t if t % 2 else t + 1, 3)
        self.conv = nn.Conv1d(1, 1, kernel_size=k, padding=(k - 1) // 2, bias=False)
        self.conv_local = nn.Conv1d(1, 1, kernel_size=k, padding=(k - 1) // 2, bias=False)
        self.local_pool = nn.AdaptiveAvgPool2d(self.local_size)
        self.global_pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        b, c, h, w = x.shape
        ls = self.local_size
        local = self.local_pool(x)
        glob = self.global_pool(local)
        t_local = local.view(b, c, -1).transpose(-1, -2).reshape(b, 1, -1)
        t_glob = glob.view(b, c, -1).transpose(-1, -2)
        y_local = self.conv_local(t_local)
        y_glob = self.conv(t_glob)
        att_local = y_local.reshape(b, ls * ls, c).transpose(-1, -2).reshape(b, c, ls, ls).sigmoid()
        att_glob = y_glob.transpose(-1, -2).reshape(b, c, 1, 1).sigmoid().expand(-1, -1, ls, ls)
        att = att_glob * (1 - self.local_weight) + att_local * self.local_weight
        return x * F.interpolate(att, size=(h, w), mode='nearest')


class Bottleneck_MLCA(Bottleneck):
    def __init__(self, c1, c2, shortcut=True, g=1, k=(3, 3), e=0.5):
        super().__init__(c1, c2, shortcut, g, k, e)
        self.mlca = MLCA(c2)

    def forward(self, x):
        y = self.mlca(self.cv2(self.cv1(x)))
        return x + y if self.add else y


class C2f_MLCA(C2f):
    def __init__(self, c1, c2, n=1, shortcut=False, g=1, e=0.5):
        super().__init__(c1, c2, n, shortcut, g, e)
        self.m = nn.ModuleList(
            Bottleneck_MLCA(self.c, self.c, shortcut, g, k=((3, 3), (3, 3)), e=1.0) for _ in range(n))


# registrar (mismo nombre de módulo que en la máquina local: los .pt se pueden cargar en los dos lados)
for cls in (MLCA, Bottleneck_MLCA, C2f_MLCA):
    cls.__module__ = 'ultralytics.nn.modules.block'
    cls.__qualname__ = cls.__name__
    setattr(block, cls.__name__, cls)
    setattr(tasks, cls.__name__, cls)

assert 'C2fPSA' in inspect.getsource(tasks.parse_model), 'esta version de ultralytics cambio parse_model'
if not getattr(tasks.parse_model, '_mlca_v8', False):
    _original_parse_model = tasks.parse_model
    def parse_model(d, ch, verbose=True):
        saved = tasks.C2fPSA
        tasks.C2fPSA = C2f_MLCA
        try:
            return _original_parse_model(d, ch, verbose)
        finally:
            tasks.C2fPSA = saved
    parse_model._mlca_v8 = True
    tasks.parse_model = parse_model
print('C2f-MLCA registrado')

## 7 · Arquitectura y prueba

In [ ]:
from collections import Counter
from ultralytics import YOLO

src = os.path.join(os.path.dirname(ultralytics.__file__), 'cfg', 'models', 'v8', 'yolov8.yaml')
out, in_head, n = [], False, 0
for ln in open(src).read().splitlines():
    if ln.startswith('head:'):
        in_head = True
    if in_head and ', C2f, [' in ln:
        ln, n = ln.replace(', C2f, [', ', C2f_MLCA, ['), n + 1
    if ln.startswith('nc:'):
        ln = 'nc: 3'
    out.append(ln)
assert n == 4
MLCA_YAML = '/content/yolov8m-mlca.yaml'      # el nombre empieza con 'yolov8m' -> escala m
open(MLCA_YAML, 'w').write('\n'.join(out) + '\n')


def make_model(variant):
    if variant == 'base':
        return YOLO('yolov8m.pt')               # exactamente como Belén
    m = YOLO(MLCA_YAML)
    m.load('yolov8m.pt')                       # COCO en todo menos el MLCA
    return m


for v in VARIANTES:
    m = make_model(v)
    k = Counter(type(x).__name__ for x in m.model.modules())
    bb = Counter(type(x).__name__ for x in m.model.model[:10].modules())
    print(f"{v:5s} C2f={k['C2f']} C2f_MLCA={k['C2f_MLCA']} MLCA={k['MLCA']} | MLCA en backbone={bb['MLCA']}")
    if v == 'mlca':
        assert k['C2f_MLCA'] == 4 and k['MLCA'] == 8 and bb['MLCA'] == 0
# esperado:  base C2f=8 C2f_MLCA=0 MLCA=0  |  mlca C2f=4 C2f_MLCA=4 MLCA=8, 0 en el backbone

## 8 · Entrenar

Una corrida por variante y seed, intercaladas (`base_s0`, `mlca_s0`, `base_s1`, ...). Las corridas terminadas quedan marcadas en Drive y se saltean si volvés a ejecutar la celda.

In [ ]:
import time
RUNS = os.path.join(DRIVE_DIR, 'runs')
for seed in SEEDS:
    for v in VARIANTES:
        name = f'{v}_s{seed}'
        done = os.path.join(RUNS, name, 'TERMINADO')
        if os.path.exists(done):
            print('ya estaba:', name)
            continue
        t0 = time.time()
        m = make_model(v)
        m.train(data=DATA_YAML, epochs=EPOCHS, batch=BATCH, seed=seed, device=0, project=RUNS,
                name=name, exist_ok=True, plots=True, **HER_ARGS)
        best = os.path.join(RUNS, name, 'weights', 'best.pt')
        if os.path.exists(best):
            open(done, 'w').write('ok')
            print(f'>>> {name} listo en {(time.time() - t0) / 60:.1f} min')
        else:
            print(f'!!! {name} terminó sin best.pt: NO se marca como terminada, volvé a ejecutar')

## 9 · Evaluar

Para cada corrida (`best.pt`, elegido por mAP50-95 en valid, como Belén):

* **mAP** con `model.val` sobre su valid (24 fotos) y su test (31), como en su presentación.
* **Especie por placa y conteo** con su configuración: conf 0,5, NMS 0,45, especie = voto mayoritario. La especie verdadera sale del nombre del archivo.

Si subís su `best_v8.pt` a la carpeta de Drive, se evalúa también como referencia.

In [ ]:
import os
print(os.listdir(os.path.join(RUNS, 'mlca_s2', 'weights')))

In [ ]:
import glob, json, re
import numpy as np
import pandas as pd

NA = ['col', 'osa', 'sau']
def especie(nombre):
    n = nombre.lower()
    return 'col' if 'ecoli' in n else 'sau' if 'saureus' in n else 'osa' if ('paureginosa' in n or 'paeruginosa' in n) else None

def placas(model, split):
    err, ok = [], []
    for f in sorted(glob.glob(f'/content/dataset_v5/{split}/images/*.jpg')):
        lab = f.replace('/images/', '/labels/')[:-4] + '.txt'
        n_gt = sum(1 for l in open(lab) if len(l.split()) == 5)
        r = model.predict(f, imgsz=640, conf=0.5, iou=0.45, verbose=False)[0]
        cl = r.boxes.cls.cpu().numpy().astype(int)
        vote = NA[int(np.bincount(cl, minlength=3).argmax())] if len(cl) else None
        err.append(len(cl) - n_gt)
        ok.append(vote == especie(os.path.basename(f)))
    return float(np.mean(ok)), float(np.mean(np.abs(err)))

# --- diagnóstico: qué hay realmente en Drive ---
for d in sorted(glob.glob(os.path.join(RUNS, '*_s*'))):
    wd = os.path.join(d, 'weights')
    print(os.path.basename(d),
          '| TERMINADO:', os.path.exists(os.path.join(d, 'TERMINADO')),
          '|', os.listdir(wd) if os.path.isdir(wd) else 'SIN carpeta weights')

# --- lista de modelos a evaluar, solo con archivos que existen ---
runs = []
for d in sorted(glob.glob(os.path.join(RUNS, '*_s*'))):
    nombre = os.path.basename(d)
    if not re.fullmatch(r'(base|mlca)_s\d+', nombre):
        continue
    w = os.path.join(d, 'weights', 'best.pt')
    if os.path.exists(os.path.join(d, 'TERMINADO')) and os.path.exists(w):
        runs.append((nombre, w))
    else:
        print('SALTEADA (falta TERMINADO o best.pt):', nombre)

HER = os.path.join(DRIVE_DIR, 'best_v8.pt')
if os.path.exists(HER):
    runs.insert(0, ('ella_colab', HER))
else:
    print('best_v8.pt no está en', DRIVE_DIR, '-> se evalúa sin la referencia')

assert runs, 'No hay modelos para evaluar: revise el diagnóstico de arriba.'

# --- evaluación ---
rows = []
for name, w in runs:
    m = YOLO(w)
    row = {'corrida': name, 'variante': name.rsplit('_s', 1)[0] if re.fullmatch(r'(base|mlca)_s\d+', name) else name}
    for split, sp in (('val', 'valid'), ('test', 'test')):
        r = m.val(data=DATA_YAML, split=split, imgsz=640, batch=16, conf=0.001, iou=0.7, plots=False, verbose=False)
        row[f'{split}_mAP50'] = float(r.box.map50)
        row[f'{split}_mAP50_95'] = float(r.box.map)
        row[f'{split}_especie'], row[f'{split}_conteo_MAE'] = placas(m, sp)
    rows.append(row)
    print(name, {k: round(v, 3) for k, v in row.items() if isinstance(v, float)})

df = pd.DataFrame(rows)
df.to_csv(os.path.join(DRIVE_DIR, 'resultados_colab.csv'), index=False)
cols = ['val_mAP50', 'test_mAP50', 'test_mAP50_95', 'test_especie', 'test_conteo_MAE']
print('\nPromedio ± desvío por variante:')
display(df.groupby('variante')[cols].agg(['mean', 'std']).round(3))

## 10 · Bajar los resultados

Descarga `resultados_colab.csv` (y, si querés comparar en la máquina local, los `best.pt`, que quedan en `MyDrive/yolov8m_mlca/runs/*/weights/`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files
files.download(os.path.join(DRIVE_DIR, 'resultados_colab.csv'))